In [1]:
# Import packages and modules
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow_decision_forests as tfdf

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

2026-08-12 10:20:24.896543: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-12 10:20:25.327329: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-08-12 10:20:30.229659: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
# Check the version of TensorFlow Decision Forests
print("Found TensorFlow Decision Forests v" + tfdf.__version__)

Found TensorFlow Decision Forests v1.9.1


In [3]:
datasetPath = 'dataset/firstorder/kernel5-radius5/dataset.csv'

In [4]:
# Parameters
datasetPath = "dataset/gldm/kernel6/1.dataset.csv"


In [5]:
# train/validation/test = 70/10/20
dataset = pd.read_csv(datasetPath)

train_data, temp_data = train_test_split(dataset, test_size=0.3, random_state=42)
validation_data, test_data = train_test_split(temp_data, test_size=2/3, random_state=42)

In [6]:
# Convert the dataset into a TensorFlow dataset.
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_data, label="label"
)         
val_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    validation_data, label="label"
)
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    test_data, label="label"
)

2026-08-12 10:20:43.968621: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-08-12 10:20:43.970188: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [7]:
import keras

In [8]:
%%time

# Train an SVM model with class weight.
svm_model = SVC(kernel="rbf", probability=True, random_state=42, class_weight="balanced")
svm_model.fit(train_data.drop(columns=["label"]), train_data["label"])

CPU times: user 13.5 ms, sys: 932 µs, total: 14.4 ms
Wall time: 15.6 ms


SVC(class_weight='balanced', probability=True, random_state=42)

In [9]:
# Evaluate the model with sklearn
from sklearn.metrics import accuracy_score

X_val = validation_data.drop(columns=["label"])
y_true = validation_data["label"].astype(int).to_numpy()
y_pred = svm_model.predict(X_val)

accuracy = accuracy_score(y_true, y_pred)
print(f"accuracy: {accuracy:.4f}")

accuracy: 0.9219


In [10]:
# Model Summary (sklearn style)
print(svm_model)
print(f"kernel: {svm_model.kernel}")
print(f"C: {svm_model.C}")
print(f"gamma: {svm_model.gamma}")

SVC(class_weight='balanced', probability=True, random_state=42)
kernel: rbf
C: 1.0
gamma: scale


In [11]:
# Model features
feature_names = train_data.drop(columns=["label"]).columns.tolist()
print("features:")
print(feature_names)

features:
['DependenceEntropy', 'DependenceNonUniformity', 'DependenceNonUniformityNormalized', 'DependenceVariance', 'GrayLevelNonUniformity', 'GrayLevelVariance', 'HighGrayLevelEmphasis', 'LargeDependenceEmphasis', 'LargeDependenceHighGrayLevelEmphasis', 'LargeDependenceLowGrayLevelEmphasis', 'LowGrayLevelEmphasis', 'SmallDependenceEmphasis', 'SmallDependenceHighGrayLevelEmphasis', 'SmallDependenceLowGrayLevelEmphasis']


In [12]:
# Feature importance (not available for SVC with RBF kernel)
print("SVC with RBF kernel does not provide feature importances.")

SVC with RBF kernel does not provide feature importances.


In [13]:
# Model self evaluation (sklearn info)
print("number of support vectors:", svm_model.support_vectors_.shape[0])
print("support vectors per class:", svm_model.n_support_)

number of support vectors: 155
support vectors per class: [130  25]


In [14]:
# Training logs (not available for sklearn SVC)
print("Training logs are not available for sklearn SVC.")

Training logs are not available for sklearn SVC.


Calculate the score of our hold-out validation dataset

In [15]:
X_val = validation_data.drop(columns=["label"])
y_true = validation_data["label"].astype(int).to_numpy()
pos_probs = svm_model.predict_proba(X_val)[:, 1]

from sklearn.metrics import roc_auc_score
ROC_AUC = roc_auc_score(y_true, pos_probs)
print("The ROC AUC score is %.5f" % ROC_AUC )

The ROC AUC score is 0.97778


In [16]:
# Compute binary classification metrics with sklearn
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    log_loss,
    brier_score_loss,
 )

# Get predictions (probabilities or class labels)
X_val = test_data.drop(columns=["label"])
y_true = test_data["label"].astype(int).to_numpy()
pos_probs = svm_model.predict_proba(X_val)[:, 1]
y_pred = (pos_probs >= 0.5).astype(int)

# Core metrics
metrics = {}
metrics["accuracy"] = accuracy_score(y_true, y_pred)
metrics["precision"] = precision_score(y_true, y_pred, zero_division=0)
metrics["recall"] = recall_score(y_true, y_pred, zero_division=0)
metrics["f1"] = f1_score(y_true, y_pred, zero_division=0)
metrics["mcc"] = matthews_corrcoef(y_true, y_pred)

# Probabilistic metrics
metrics["roc_auc"] = roc_auc_score(y_true, pos_probs)
metrics["pr_auc"] = average_precision_score(y_true, pos_probs)
metrics["log_loss"] = log_loss(y_true, pos_probs, labels=[0,1])
metrics["brier_score"] = brier_score_loss(y_true, pos_probs)

# Confusion matrix and detailed report
cm = confusion_matrix(y_true, y_pred, labels=[0,1])
report = classification_report(y_true, y_pred, digits=4)

print("Sklearn binary metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")
print("\nConfusion matrix:\n", cm)
print("\nClassification report:\n", report)

Sklearn binary metrics:
accuracy: 0.8992
precision: 0.5294
recall: 0.6429
f1: 0.5806
mcc: 0.5272
roc_auc: 0.9304
pr_auc: 0.6101
log_loss: 0.2054
brier_score: 0.0638

Confusion matrix:
 [[107   8]
 [  5   9]]

Classification report:
               precision    recall  f1-score   support

           0     0.9554    0.9304    0.9427       115
           1     0.5294    0.6429    0.5806        14

    accuracy                         0.8992       129
   macro avg     0.7424    0.7866    0.7617       129
weighted avg     0.9091    0.8992    0.9034       129

